# Stateful Agent Orchestration, Write-Ahead Log (WAL) & Crash Replay

> **Lab 03 & Phase 04 Companion Exercise**  
> **Core Architecture**: Stateful ReAct Loop + Write-Ahead Log (WAL) Event Sourcing + Deterministic Crash Recovery + Time-Travel Debugger.

In production cloud environments, long-running agent workflows are frequently interrupted by Kubernetes pod evictions, out-of-memory (OOM) events, and transient network partitions. If agent state is stored in an ephemeral Python dictionary, a mid-workflow crash forces a naive restart from scratch, causing the **Double-Spend Problem**: non-idempotent operations (such as payments or database writes) are executed twice! This notebook provides a complete implementation of an **Event-Sourced Write-Ahead Log (WAL)**, simulates mid-turn crashes, and demonstrates deterministic state rehydration and replay.

In [ ]:
# %% [Setup] Colab & Environment Bootstrap
import os, sys
try:
    is_colab = 'google.colab' in str(get_ipython())
except NameError:
    is_colab = False

if is_colab:
    print('Running in Google Colab.')
    repo_dir = '/content/repo'
    if not os.path.exists(repo_dir):
        print('Cloning repository...')
        !git clone --depth 1 https://github.com/dip4k/senior-engineer-to-ai-engineer-roadmap.git /content/repo
    %cd /content/repo
    !pip install -q --no-warn-conflicts -r agent-forge/requirements.txt -r notebooks/requirements-notebooks.txt
    if os.path.abspath('agent-forge') not in sys.path:
        sys.path.insert(0, os.path.abspath('agent-forge'))
else:
    # Local environment setup: locate repo root and agent-forge
    repo_root = os.path.abspath('..') if os.path.exists('../agent-forge') else os.path.abspath('.')
    af_path = os.path.join(repo_root, 'agent-forge')
    if os.path.exists(af_path) and af_path not in sys.path:
        sys.path.insert(0, af_path)
    print(f'Running locally with agent-forge in path: {af_path}')

## 1. Architectural Architecture: The Fragility of Ephemeral Agent State

Standard prototype agents run in an unbuffered Python loop:
```text
while not finished:
    thought = llm.think()
    action = llm.decide_action()
    result = execute(action)
    messages.append(result)
```

If the container is killed after `execute(action)` but before `messages.append(result)`, the execution history is obliterated. A restart executes the action again.

The **Write-Ahead Log (WAL) Invariant** guarantees resilience:
- **Commit Before Mutation**: Every state transition (thought, decision, tool invocation, completion) is persisted to an immutable, append-only `EventStore` *before* the next action begins.
- **Deterministic Rehydration**: On crash recovery, the session reconstructs its memory by folding over the event stream.

## 2. Pydantic Domain Models for Events & Sessions

We define strict schemas for events and sessions matching `agent_forge.runtime`.

In [ ]:
import time, uuid, json
from typing import List, Dict, Any, Optional, Literal
from pydantic import BaseModel, Field

# ---------------------------------------------------------------------------
# Domain Models for Messages & Tool Calls
# ---------------------------------------------------------------------------
class ToolCall(BaseModel):
    id: str = Field(default_factory=lambda: f'call_{uuid.uuid4().hex[:8]}')
    name: str
    arguments: Dict[str, Any]

class Message(BaseModel):
    role: Literal['system', 'user', 'assistant', 'tool']
    content: Optional[str] = None
    tool_calls: Optional[List[ToolCall]] = None
    tool_call_id: Optional[str] = None
    timestamp: float = Field(default_factory=time.time)

# ---------------------------------------------------------------------------
# Immutable Event Model for Write-Ahead Log
# ---------------------------------------------------------------------------
class AgentEvent(BaseModel):
    event_id: str = Field(default_factory=lambda: f'evt_{uuid.uuid4().hex[:10]}')
    session_id: str
    turn_index: int
    event_type: Literal[
        'session_started',
        'turn_started',
        'model_decision',
        'tool_executing',
        'tool_completed',
        'checkpoint_saved',
        'session_completed'
    ]
    payload: Dict[str, Any] = Field(default_factory=dict)
    timestamp: float = Field(default_factory=time.time)

class AgentSession(BaseModel):
    session_id: str
    tenant_id: str = 'default_tenant'
    user_id: str = 'default_user'
    status: Literal['running', 'paused', 'completed', 'failed'] = 'running'
    current_turn: int = 0
    messages: List[Message] = Field(default_factory=list)
    executed_tools: List[str] = Field(default_factory=list)

print('Domain models initialized successfully.')

## 3. Append-Only EventStore & Rehydration Engine

The `EventStore` provides two critical operations:
1. `append(event)`: Persists an immutable state transition.
2. `rehydrate_session(session_id)`: Replays historical events to restore the session state.

In [ ]:
# ---------------------------------------------------------------------------
# Thread-Safe Append-Only EventStore
# ---------------------------------------------------------------------------
class EventStore:
    def __init__(self):
        self._events: List[AgentEvent] = []
        self._checkpoints: Dict[str, Dict[str, Any]] = {}

    def append(self, event: AgentEvent) -> None:
        self._events.append(event)

    def get_events(self, session_id: str) -> List[AgentEvent]:
        return [e for e in self._events if e.session_id == session_id]

    def save_checkpoint(self, session: AgentSession) -> None:
        self._checkpoints[session.session_id] = session.model_dump()
        self.append(AgentEvent(
            session_id=session.session_id,
            turn_index=session.current_turn,
            event_type='checkpoint_saved',
            payload={'turn': session.current_turn, 'status': session.status}
        ))

    def rehydrate_session(self, session_id: str) -> Optional[AgentSession]:
        events = self.get_events(session_id)
        if not events:
            return None

        # Initialize base session
        session = AgentSession(
            session_id=session_id,
            tenant_id=events[0].payload.get('tenant_id', 'default_tenant'),
            user_id=events[0].payload.get('user_id', 'default_user')
        )

        # Fold over event stream to reconstruct state
        for event in events:
            session.current_turn = max(session.current_turn, event.turn_index)
            
            if event.event_type == 'session_started':
                goal = event.payload.get('goal', '')
                session.messages.append(Message(role='user', content=goal))
            elif event.event_type == 'model_decision':
                thought = event.payload.get('thought', '')
                tool_name = event.payload.get('tool_name')
                tool_args = event.payload.get('arguments', {})
                tool_calls = [ToolCall(name=tool_name, arguments=tool_args)] if tool_name else None
                session.messages.append(Message(role='assistant', content=thought, tool_calls=tool_calls))
            elif event.event_type == 'tool_completed':
                tool_name = event.payload.get('tool_name', 'unknown')
                session.executed_tools.append(tool_name)
                session.messages.append(Message(
                    role='tool',
                    tool_call_id=event.payload.get('tool_call_id'),
                    content=json.dumps(event.payload.get('result', {}))
                ))
            elif event.event_type == 'session_completed':
                session.status = 'completed'
                
        return session

store = EventStore()
print('EventStore initialized.')

## 4. Multi-Turn ReAct Workflow with Write-Ahead Logging

We simulate a realistic 3-turn customer resolution workflow:
- **Turn 0**: Inbound user request: *"Please investigate duplicate charge on order 9182."*
- **Turn 1**: Agent queries order details via `order_get_order`.
- **Turn 2**: Agent inspects charges, identifies duplicate, and issues refund via `payment_issue_refund` (49.00 USD).
- **Turn 3**: Agent synthesizes final customer explanation.

In [ ]:
# ---------------------------------------------------------------------------
# Simulated Tools with Mutation Tracking
# ---------------------------------------------------------------------------
execution_counters = {'order_get_order': 0, 'payment_issue_refund': 0}

def tool_order_get_order(order_id: str):
    execution_counters['order_get_order'] += 1
    return {'order_id': order_id, 'items': ['Pro Cloud Subscription'], 'status': 'CHARGED'}

def tool_payment_issue_refund(transaction_id: str, amount: float):
    execution_counters['payment_issue_refund'] += 1
    return {'refund_id': 'ref_9182_dup', 'amount': amount, 'status': 'PROCESSED'}

# ---------------------------------------------------------------------------
# Execute Turn 0 & Turn 1
# ---------------------------------------------------------------------------
session_id = 'sess_prod_9942'

# Turn 0: Session Start
store.append(AgentEvent(
    session_id=session_id,
    turn_index=0,
    event_type='session_started',
    payload={'goal': 'Investigate duplicate charge on order 9182', 'tenant_id': 'acme_corp', 'user_id': 'support_agent'}
))

# Turn 1: Model Decision & Tool Execution
store.append(AgentEvent(
    session_id=session_id,
    turn_index=1,
    event_type='model_decision',
    payload={'thought': 'I need to check the order status first.', 'tool_name': 'order_get_order', 'arguments': {'order_id': '9182'}}
))
res_1 = tool_order_get_order('9182')
store.append(AgentEvent(
    session_id=session_id,
    turn_index=1,
    event_type='tool_completed',
    payload={'tool_name': 'order_get_order', 'result': res_1}
))

# Turn 2: Financial Mutation (Refund Issued!)
store.append(AgentEvent(
    session_id=session_id,
    turn_index=2,
    event_type='model_decision',
    payload={'thought': 'Duplicate detected. Issuing 49.00 USD refund.', 'tool_name': 'payment_issue_refund', 'arguments': {'transaction_id': 'tx_dup', 'amount': 49.0}}
))
res_2 = tool_payment_issue_refund('tx_dup', 49.0)
store.append(AgentEvent(
    session_id=session_id,
    turn_index=2,
    event_type='tool_completed',
    payload={'tool_name': 'payment_issue_refund', 'result': res_2}
))

print(f'Events logged in WAL: {len(store.get_events(session_id))}')
print(f'Tool executions so far: {execution_counters}')

## 5. Simulating a Sudden Pod Eviction / OOM Crash

Now, disaster strikes: immediately following Turn 2, the Kubernetes node terminates the pod. Any in-memory variables are wiped clean.

Notice what would happen if a naive orchestrator restarted from Turn 0 without WAL rehydration:
- It would observe that the refund is needed.
- It would execute `payment_issue_refund` a second time!
- `execution_counters['payment_issue_refund']` would equal 2 (double refund!).

In [ ]:
# Simulate process termination
active_session = None
print('=== SIMULATED POD CRASH: PROCESS TERMINATED ===')
print(f'In-memory session state: {active_session}')
print(f'Preserved events in persistent EventStore: {len(store.get_events(session_id))}')

## 6. Deterministic Crash Recovery & Resume

When the replacement pod initializes, it connects to the `EventStore`, calls `rehydrate_session`, and reconstructs the exact state. It notices that `payment_issue_refund` was **already executed** and advances immediately to Turn 3 (final summary).

In [ ]:
# ---------------------------------------------------------------------------
# Crash Recovery Rehydration
# ---------------------------------------------------------------------------
recovered_session = store.rehydrate_session(session_id)
print('Rehydrated Session Status:')
print(f'  Session ID: {recovered_session.session_id}')
print(f'  Current Turn: {recovered_session.current_turn}')
print(f'  Executed Tools: {recovered_session.executed_tools}')
print(f'  Message Count: {len(recovered_session.messages)}')

# Assert that refund tool is in executed_tools
assert 'payment_issue_refund' in recovered_session.executed_tools, 'Recovery error: refund not tracked!'

# Resume Turn 3: Final Answer synthesis without re-executing tools
turn_3_event = AgentEvent(
    session_id=session_id,
    turn_index=3,
    event_type='session_completed',
    payload={'final_response': 'Duplicate charge refunded successfully. Reference: ref_9182_dup.'}
)
store.append(turn_3_event)

print(f'\nWorkflow resumed and completed!')
print(f'Total refund executions: {execution_counters["payment_issue_refund"]} (Expected: exactly 1!)')

## 7. Time-Travel Debugger & Event Stream Visualizer

Because every state transition is recorded in chronological order, we can build a **Time-Travel Debugger** to inspect the session state at any arbitrary turn.

In [ ]:
import matplotlib.pyplot as plt

events = store.get_events(session_id)
turn_indices = [e.turn_index for e in events]
event_names = [f"{e.turn_index}: {e.event_type}" for e in events]
timestamps = [e.timestamp - events[0].timestamp for e in events]

fig, ax = plt.subplots(figsize=(11, 4.5))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

for i, (name, t) in enumerate(zip(event_names, timestamps)):
    ax.scatter(t, i, color=colors[i % len(colors)], s=120, zorder=3)
    ax.hlines(i, 0, t, linestyles='dashed', alpha=0.4, color='gray')
    ax.text(t + 0.001, i, f'  {name}', va='center', fontsize=9, fontweight='bold')

ax.set_yticks(range(len(event_names)))
ax.set_yticklabels([e.event_type for e in events])
ax.set_xlabel('Elapsed Time from Start (Seconds)', fontsize=10)
ax.set_title('Write-Ahead Log (WAL) Event Timeline & Crash Recovery Trace', fontsize=12, fontweight='bold')
ax.grid(True, linestyle=':', alpha=0.6)
plt.subplots_adjust(left=0.22, right=0.95, top=0.9, bottom=0.15)
plt.show()

## 8. Interactive Katas & Idempotency Challenge

### Kata: Idempotency Key Validation
In enterprise systems, even if an agent retries an action, tools must support an `idempotency_key`. If a tool call with the same idempotency key is received twice, the tool returns the cached output.

In [ ]:
# ---------------------------------------------------------------------------
# Kata Solution: Idempotent Tool Execution Gate
# ---------------------------------------------------------------------------
class IdempotentToolRunner:
    def __init__(self):
        self.cache: Dict[str, Any] = {}
        self.raw_executions = 0

    def execute(self, key: str, action: str, amount: float) -> Dict[str, Any]:
        if key in self.cache:
            print(f'  [CACHE HIT] Returning cached result for idempotency key {key}')
            return self.cache[key]
        self.raw_executions += 1
        result = {'status': 'COMPLETED', 'action': action, 'amount': amount, 'key': key}
        self.cache[key] = result
        return result

runner = IdempotentToolRunner()
r1 = runner.execute('idemp_key_101', 'refund', 49.0)
r2 = runner.execute('idemp_key_101', 'refund', 49.0)
assert runner.raw_executions == 1, 'Idempotency violation: executed twice!'
print('Idempotency validation verified: Duplicate execution was safely intercepted.')

## 9. Summary & Production Architecture Checklist

- **Write-Ahead Log First**: Always commit decisions and turn transitions to persistent storage before calling external tools.
- **Avoid Ephemeral State**: Treat in-memory dictionaries as expendable caches.
- **Deterministic State Folding**: Session memory is a fold over the immutable event stream.
- **Idempotent Mutations**: Complement WAL recovery with cryptographic idempotency keys on all mutating tools.